In [3]:
# =============================================================
# HC-WCI AI FRAMEWORK — RECTIFIED VERSION
# Wright State University | Amir Hasan Khan
#
# KEY FIXES FROM SANITY CHECK REPORT:
#   FIX 1: Cohort-size weighting added (sample_weight = Total_People)
#           Each row now contributes proportionally to its population size
#   FIX 2: Minimum cohort filter (>= 100 people) applied before training
#           to remove degenerate 0%/100% mortgage rate spikes
#   FIX 3: Rp outliers winsorized at 95th percentile before modelling
#   FIX 4: FAIR baseline comparison: 4-model grid that separates
#           feature contribution from algorithm contribution
#   FIX 5: "Accuracy" replaced with "Coefficient of Determination (R²)"
#           throughout. R² is NOT an accuracy metric.
#   FIX 6: Structural Rp/M linkage explicitly disclosed in output
#   NOTE:   Uses Mortgaged_Ownership_Rate (renamed column from v2 dataset)
#           If running on original dataset, update column name to
#           Mortgage_Rate_Pct at line marked *** COLUMN NAME ***
# =============================================================

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

print("=" * 55)
print("  HC-WCI Machine Learning Framework — Rectified")
print("=" * 55)

# ── 1. LOAD DATA ─────────────────────────────────────────────
# If using the rectified v2 dataset, the target column is
# 'Mortgaged_Ownership_Rate'. If using the original dataset,
# change TARGET_COL to 'Mortgage_Rate_Pct'.

try:
    df = pd.read_csv('HCWCI_Master_Dataset_v2.csv')
    TARGET_COL = 'Mortgaged_Ownership_Rate'       # *** COLUMN NAME ***
    RP_COL     = 'Rent_Performance_Ratio'
    print(f"Loaded v2 dataset. Total cohorts: {len(df):,}\n")
except FileNotFoundError:
    # Fallback: use original dataset with original column names
    df = pd.read_csv('HCWCI_Master_Dataset.csv')
    TARGET_COL = 'Mortgage_Rate_Pct'              # *** COLUMN NAME ***
    RP_COL     = 'Rent_Performance_Ratio'
    print(f"Loaded original dataset. Total cohorts: {len(df):,}\n")
    print("WARNING: Original dataset has known Rp/M structural linkage.")
    print("         Re-run extraction code first to get rectified data.\n")


# ── 2. PREPROCESSING ─────────────────────────────────────────
edu_map = {
    "1_Low (No/Primary)":           1,
    "2_Some High School":           2,
    "3_HS Graduate":                3,
    "4_Some College":               4,
    "5_College+ (Bachelor or Higher)": 5
}
df['Edu_Tier'] = df['Education_Group'].map(edu_map)

# FIX 2: minimum cohort filter
df = df[df['Total_People'] >= 100].copy()

# FIX 3: winsorize Rp at 95th percentile
rp_cap = df[RP_COL].quantile(0.95)
df['Rp_Win'] = df[RP_COL].clip(upper=rp_cap)

df = df.dropna(
    subset=['Years_in_US', 'Edu_Tier', 'Rp_Win', 'Avg_Income', TARGET_COL]
)

print(f"After quality filters: {len(df):,} cohorts")
print(f"  Rp winsorized cap (95th pct): {rp_cap:.3f}")
print(f"  Total weighted population:    {df['Total_People'].sum():,.0f}\n")


# ── 3. DEFINE FEATURES AND TARGET ───────────────────────────
y  = df[TARGET_COL]
sw = df['Total_People']    # FIX 1: population-proportional sample weights

X_hcwci  = df[['Years_in_US', 'Edu_Tier', 'Rp_Win']]   # HC-WCI features
X_income  = df[['Avg_Income']]                           # income only
X_no_rp   = df[['Years_in_US', 'Edu_Tier']]             # HC-WCI minus Rp


# ── 4. 80/20 SPLIT — same seed for all models ────────────────
(X_hc_tr, X_hc_te,
 y_tr, y_te,
 sw_tr, sw_te) = train_test_split(X_hcwci, y, sw,
                                  test_size=0.2, random_state=42)

X_inc_tr, X_inc_te, _, _ = train_test_split(
    X_income, y, test_size=0.2, random_state=42)

X_nrp_tr, X_nrp_te, _, _ = train_test_split(
    X_no_rp, y, test_size=0.2, random_state=42)

print(f"Training set: {len(y_tr):,} cohorts")
print(f"Testing set:  {len(y_te):,} cohorts\n")


# ── 5. TRAIN ALL FOUR MODELS ─────────────────────────────────

# MODEL A — HC-WCI Random Forest (3 features, population-weighted)
print("Training Model A: HC-WCI Random Forest (weighted)...")
rf_hc = RandomForestRegressor(n_estimators=100, random_state=42)
rf_hc.fit(X_hc_tr, y_tr, sample_weight=sw_tr)  # FIX 1
pred_A = rf_hc.predict(X_hc_te)

# MODEL B — Income-only Random Forest (FIX 4: fair comparison, same algorithm)
print("Training Model B: Income-only Random Forest...")
rf_inc = RandomForestRegressor(n_estimators=100, random_state=42)
rf_inc.fit(X_inc_tr, y_tr)
pred_B = rf_inc.predict(X_inc_te)

# MODEL C — Income-only Linear Regression (original baseline for reference)
print("Training Model C: Income-only Linear Regression (original baseline)...")
lr_inc = LinearRegression()
lr_inc.fit(X_inc_tr, y_tr)
pred_C = lr_inc.predict(X_inc_te)

# MODEL D — HC-WCI without Rp (Years + Education only)
# This isolates how much Years and Education contribute independently of Rp
print("Training Model D: HC-WCI (Years + Education, no Rp)...")
rf_nrp = RandomForestRegressor(n_estimators=100, random_state=42)
rf_nrp.fit(X_nrp_tr, y_tr)
pred_D = rf_nrp.predict(X_nrp_te)

print()


# ── 6. EVALUATE ──────────────────────────────────────────────
def evaluate(name, y_true, y_pred):
    r2   = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    return {'Model': name, 'R² (CoD)': round(r2, 4), 'RMSE': round(rmse, 4)}

results = [
    evaluate("HC-WCI RF (Years + Edu + Rp, weighted)",    y_te, pred_A),
    evaluate("Income-only RF  (fair baseline)",           y_te, pred_B),
    evaluate("Income-only Linear (original baseline)",    y_te, pred_C),
    evaluate("HC-WCI RF  (Years + Edu, NO Rp)",          y_te, pred_D),
]
results_df = pd.DataFrame(results)


# ── 7. PRINT RESULTS ─────────────────────────────────────────
print("=" * 55)
print("  FINAL MODEL PERFORMANCE — COEFFICIENT OF DETERMINATION")
print("  (R² = proportion of variance explained, NOT 'accuracy')")
print("=" * 55)
print(results_df.to_string(index=False))

print()
print("=" * 55)
print("  HC-WCI FEATURE IMPORTANCE (Gini Impurity)")
print("=" * 55)
feat_names = ['Years_in_US', 'Edu_Tier', 'Rp_Win']
for name, imp in zip(feat_names, rf_hc.feature_importances_):
    print(f"  {name:30}: {imp * 100:.2f}%")

print()
print("=" * 55)
print("  INTERPRETATION NOTES")
print("=" * 55)
print("""
Model A vs Model C (original comparison):
  This confounds algorithm choice (RF vs Linear) with feature choice.
  Use Models A vs B for a fair feature comparison.

Model A vs Model B (fair comparison):
  Isolates the contribution of HC-WCI features over income alone,
  using the same Random Forest algorithm for both.

Model D (Years + Education, no Rp):
  Shows what HC-WCI achieves without Rp.
  R² improvement over Model B = pure Years + Education signal.

Structural disclosure (for paper Limitations section):
  Rent_Performance_Ratio is constructed from RENT (non-zero only
  for renters, OWNERSHP=2). Mortgaged_Ownership_Rate is constructed
  from OWNERSHP=1 + MORTGAGE>1 (owners). In the ORIGINAL dataset
  these shared the Total_People denominator, creating a mechanical
  negative correlation (Pearson r ≈ -0.497). The v2 dataset fixes
  this by computing Rp from Total_Renters only. The 80.17% feature
  importance in the original code was partly mechanical; the
  corrected importance is reported above.
""")

  HC-WCI Machine Learning Framework — Rectified
Loaded v2 dataset. Total cohorts: 30,397

After quality filters: 30,397 cohorts
  Rp winsorized cap (95th pct): 1.766
  Total weighted population:    166,261,329

Training set: 24,317 cohorts
Testing set:  6,080 cohorts

Training Model A: HC-WCI Random Forest (weighted)...
Training Model B: Income-only Random Forest...
Training Model C: Income-only Linear Regression (original baseline)...
Training Model D: HC-WCI (Years + Education, no Rp)...

  FINAL MODEL PERFORMANCE — COEFFICIENT OF DETERMINATION
  (R² = proportion of variance explained, NOT 'accuracy')
                                 Model  R² (CoD)    RMSE
HC-WCI RF (Years + Edu + Rp, weighted)   -0.0374 24.7106
       Income-only RF  (fair baseline)   -0.2287 26.8923
Income-only Linear (original baseline)    0.0559 23.5732
       HC-WCI RF  (Years + Edu, NO Rp)    0.1403 22.4950

  HC-WCI FEATURE IMPORTANCE (Gini Impurity)
  Years_in_US                   : 35.96%
  Edu_Tier        

In [13]:
pred_export = X_hc_te.copy()
pred_export['Actual_Rate']    = y_te.values
pred_export['HC_WCI_Predicted'] = pred_A
# 1. Grab the original dataframe rows matching our test set so we keep 'Country'
# (Note: if your main dataset variable is named something other than 'df', like 'final_df', change it here)
pred_export = df.loc[X_hc_te.index].copy() 

# 2. Add the ML outputs
pred_export['Actual_Rate'] = y_te.values
pred_export['HC_WCI_Predicted'] = pred_A
pred_export['ModelD_Predicted'] = pred_D

# 3. Save a fresh, fixed CSV
pred_export.to_csv('HC_WCI_Predictions_Fixed.csv', index=False)
print("Saved successfully with Country labels!")

Saved successfully with Country labels!
